In [110]:
import os
from dotenv import load_dotenv
from requests import post, get, put, delete
from requests.auth import HTTPBasicAuth
from pprint import pprint

import json

load_dotenv(".env.dev")

True

In [19]:
stripe_rk_key = os.getenv("stripeRkKey")

In [107]:
# Stripe customers API
stripe_url_customers = f"https://api.stripe.com/v1/customers"

In [108]:
# Create a customer
customer_details = {
    "name": "John Smith",
    "email": "john@gmail.com"
}
resp = post(stripe_url_customers, data=customer_details, auth=HTTPBasicAuth(stripe_rk_key, ""))

In [29]:
# Read the customer
resp = get(f"{stripe_url_customers}/cus_xxx", auth=HTTPBasicAuth(stripe_rk_key, ""))


In [32]:
# Update the user
customer_details = {
    "name": "John Smyth",
    "email": "john@mail.com"
}
resp = post(f"{stripe_url_customers}/cus_xxx", data=customer_details, auth=HTTPBasicAuth(stripe_rk_key, ""))

In [35]:
# Delete the customer
resp = delete(f"{stripe_url_customers}/cus_xxx", auth=HTTPBasicAuth(stripe_rk_key, ""))

In [46]:
# Get list of customers
resp = get(stripe_url_customers, data={"limit": 10}, auth=HTTPBasicAuth(stripe_rk_key, ""))

In [49]:
# Search customers
resp = get(f"{stripe_url_customers}/search", data={"query": "name:'John'"}, auth=HTTPBasicAuth(stripe_rk_key, ""))

In [112]:
if resp.status_code == 200:
    pprint(json.loads(resp.text))
else:
    raise NotImplementedError(resp.status_code, resp.text)

{'address': None,
 'balance': 0,
 'created': 1729873646,
 'currency': None,
 'default_source': None,
 'delinquent': False,
 'description': None,
 'discount': None,
 'email': 'john@gmail.com',
 'id': 'cus_R62R0iAWiMfpEX',
 'invoice_prefix': '6734258D',
 'invoice_settings': {'custom_fields': None,
                      'default_payment_method': None,
                      'footer': None,
                      'rendering_options': None},
 'livemode': False,
 'metadata': {},
 'name': 'John Smith',
 'object': 'customer',
 'phone': None,
 'preferred_locales': [],
 'shipping': None,
 'tax_exempt': 'none',
 'test_clock': None}


In [96]:
# Stripe payments_methods API. Particularly I'm interested in cards for now, though the whole platform looks interesting.
stripe_url_payment_methods = f"https://api.stripe.com/v1/payment_methods"

In [99]:
# Create a card
# card_details = {
#     "type": "card",
#     "card": {
#         "exp_month": 8,
#         "exp_year": 28,
#         "number": "4242424242424242",
#         "cvc": "111",
#     },
#     "billing_details": {
#         "address": {
#             "city": "Springfield",
#             "country": "US",
#             "line1": "14021, Evergreen str.",
#             "postal_code": "14001",
#             "state": "Montana",
#         },
#         "email": "john@mail.com",
#         "name": "JOHN SMITH",
#         "phone": "1234567890",
#     },
# }
card_details = {
    "type": "card",
    "billing_details": {"address": {"country": "CY"}, "name": "JOHN SMITH"},
    "card": {
        "cvc": "111",
        "exp_month": "08",
        "exp_year": "26",
        "number": "4242424242424242",
    },
}
resp = post(
    stripe_url_payment_methods, params=card_details, auth=HTTPBasicAuth(stripe_rk_key, ""), headers={"Content-Type": "application/x-www-form-urlencoded"}
)

In [100]:
if resp.status_code == 200:
    print(json.loads(resp.text))
else:
    raise NotImplementedError(resp.status_code, resp.text)

NotImplementedError: (400, '{\n  "error": {\n    "message": "Invalid object",\n    "param": "card",\n    "request_log_url": "https://dashboard.stripe.com/test/logs/req_7DeB6LAKLWU78u?t=1729871183",\n    "type": "invalid_request_error"\n  }\n}\n')

In [106]:
! . .env.dev; \
curl -i -X POST -H 'Content-Type: application/x-www-form-urlencoded' https://api.stripe.com/v1/payment_methods -u "$stripeRkKey:" \
    -d "allow_redisplay"="always" \
    -d "billing_details[address[country]]"="CY" \
    -d "billing_details[name]"="JOHN SMITH" \
    -d "card[cvc]"="111" \
    -d "card[exp_month]"="08" \
    -d "card[exp_year]"="26" \
    -d "card[number]"="4242424242424242" \
    -d "type"="card"

22206.21s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


HTTP/2 402 
server: nginx
date: Fri, 25 Oct 2024 16:09:08 GMT
content-type: application/json
content-length: 480
access-control-allow-credentials: true
access-control-allow-methods: GET, HEAD, PUT, PATCH, POST, DELETE
access-control-allow-origin: *
access-control-expose-headers: Request-Id, Stripe-Manage-Version, Stripe-Should-Retry, X-Stripe-External-Auth-Required, X-Stripe-Privileged-Session-Required
access-control-max-age: 300
cache-control: no-cache, no-store
content-security-policy: report-uri https://q.stripe.com/csp-report?p=v1%2Fpayment_methods; block-all-mixed-content; default-src 'none'; base-uri 'none'; form-action 'none'; frame-ancestors 'none'; img-src 'self'; script-src 'self' 'report-sample'; style-src 'self'
cross-origin-opener-policy-report-only: same-origin; report-to="coop"
idempotency-key: be1dce36-b4ef-42b4-8e3b-12e4557aa105
original-request: req_5scdwXGM40Z7sP
report-to: {"group":"coop","max_age":8640,"endpoints":[{"url":"https://q.stripe.com/coop-report?s=payins-

In [105]:
# does not work
! . .env.dev; \
curl -i -X POST -H 'Content-Type: application/json' https://api.stripe.com/v1/payment_methods -u "$stripeRkKey:" \
    -d '{ \
        "type": "card", \
        "billing_details": {"address": {"country": "CY"}, "name": "JOHN SMITH"}, \
        "card": { \
            "cvc": "111", \
            "exp_month": "08", \
            "exp_year": "26", \
            "number": "4242424242424242" \
        } \
    }'

22178.90s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


HTTP/2 400 
server: nginx
date: Fri, 25 Oct 2024 16:08:41 GMT
content-type: application/json
content-length: 236
access-control-allow-credentials: true
access-control-allow-methods: GET, HEAD, PUT, PATCH, POST, DELETE
access-control-allow-origin: *
access-control-expose-headers: Request-Id, Stripe-Manage-Version, Stripe-Should-Retry, X-Stripe-External-Auth-Required, X-Stripe-Privileged-Session-Required
access-control-max-age: 300
vary: Origin
x-wc: A
strict-transport-security: max-age=63072000; includeSubDomains; preload

{
  "error": {
    "message": "Invalid request (check that your POST content type is application/x-www-form-urlencoded). If you have any questions, we can help at https://support.stripe.com/.",
    "type": "invalid_request_error"
  }
}


In [ ]:
# ---------------------------------------------------------------------------------------------------

In [65]:
stripe_url_setup_intents = f"https://api.stripe.com/v1/setup_intents"

In [ ]:
# Bind the card to given customer
card_details = {
    "confirm": "true",
    "customer": "cus_R5wmv3IgsNGNUv",
    "payment_method": "pm_xxx",
    "payment_method_options": {"card": {"moto": "true"}},
    "payment_method_types": ["card"],
}

resp = post(
    stripe_url_setup_intents, data=card_details, auth=HTTPBasicAuth(stripe_rk_key, "")
)